# 91 — Load, Test & Enhance: 90-Simulate10Next_Conqueror2_Supplier_prod_per_step

Test notebook for `StrategyPipeline` from `90-Simulate10Next_Conqueror2_Supplier_prod_per_step.py`.
Each test exposes `df_s`, `pa`, `safe`, and `action` for interactive inspection.

| Test | Setup |
|------|-------|
| Test 01 | 2 Conquerors + 1 joint target + 1 solo target — prefer joint, wait alone |
| Test 02 | Same but +2 ships per player — should attack solo target immediately |

In [1]:
%run 90-Simulate10Next_Conqueror2_Supplier_prod_per_step.py

In [2]:
import copy, math, random
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}


class Obs:
    def __init__(self, planets, initial_planets=None, fleets=None,
                 next_fleet_id=100, comets=None, comet_planet_ids=None,
                 angular_velocity=0.0):
        self.planets          = [list(p) for p in planets]
        self.initial_planets  = [list(p) for p in (initial_planets if initial_planets is not None else planets)]
        self.fleets           = [list(f) for f in (fleets or [])]
        self.next_fleet_id    = next_fleet_id
        self.comets           = comets or []
        self.comet_planet_ids = comet_planet_ids or []
        self.angular_velocity = angular_velocity


def simulate_with_action(obs, action0, n_steps, current_step=0):
    snapshots = []
    for i, step in enumerate(range(current_step, current_step + n_steps)):
        snapshots.append({
            'step':    step,
            'planets': [p[:] for p in obs.planets],
            'fleets':  [f[:] for f in obs.fleets],
        })
        interpreter(obs, [action0 if i == 0 else [], []], step)
    return snapshots


def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')

    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100)
        ax.set_ylim(100, 0)
        ax.set_aspect('equal')
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y,   str(ships),         ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y+2, str(pid),            ha='center', va='center', color='red',   fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y-2, "+"+str(production), ha='center', va='center', color='white', fontsize=5, fontweight='bold', zorder=4)
        for f in snap['fleets']:
            fid, owner, x, y, angle, from_id, ships = f
            c = _COLORS.get(owner, '#888888')
            ax.plot(x, y, 'D', color=c, markersize=5, zorder=5)
            ax.text(x + 1.5, y + 1.5, str(ships), color=c, fontsize=5, zorder=6)
        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

# Test function

In [3]:
def _04_score_and_decide(attacks_with_angle: pd.DataFrame, player_id: int) -> list:
    if attacks_with_angle.empty:
        return []

    moves = []

    # Comet evasion
    awa_comets = attacks_with_angle[attacks_with_angle["nature_src"] == "comet"]
    if not awa_comets.empty:
        x_off = (awa_comets["x_src"] - GameConfig.CENTER).abs().max() or 0
        y_off = (awa_comets["y_src"] - GameConfig.CENTER).abs().max() or 0
        if max(x_off, y_off) > 45:
            moves += (
                awa_comets[awa_comets["ships_sent"] <= awa_comets["ships_min"]]
                .sort_values(["ships_sent", "step"], ascending=[False, True])
                .groupby("id_src", sort=False)
                .first()
                .reset_index()
                [["id_src", "final_angle", "ships_sent"]]
                .values.tolist()
            )
            id_to_avoid = awa_comets["id_src"].unique().tolist()
            attacks_with_angle = attacks_with_angle[~attacks_with_angle["id_src"].isin(id_to_avoid)]

    # Top-5 targets per source planet (cheapest by step then ships_sent)
    top5_ids = (
        attacks_with_angle
        .sort_values(["step", "ships_sent"])
        .groupby(["id_src", "id"], sort=False)
        .first()
        .reset_index()
        .sort_values(["step", "ships_sent"])
        .groupby("id_src", sort=False)
        .head(5)
        [["id_src", "id"]]
        .assign(is_top5=True)
    )

    # Source planet IDs owned by player (id_src is already filtered to player's planets)
    mine_src_ids = set(attacks_with_angle["id_src"].unique())

    # Classify each source as Supplier (all top5 targets are own planets) or Conqueror
    top5_with_mine = top5_ids.copy()
    top5_with_mine["target_is_mine"] = top5_with_mine["id"].isin(mine_src_ids)
    src_nature = (
        top5_with_mine
        .groupby("id_src")
        .agg(mine_count=("target_is_mine", "sum"), total_count=("target_is_mine", "count"))
        .reset_index()
    )
    src_nature["status"] = np.where(
        src_nature["mine_count"] == src_nature["total_count"], "Supplier", "Conqueror"
    )
    conqueror_ids = set(src_nature.loc[src_nature["status"] == "Conqueror", "id_src"])
    supplier_ids  = set(src_nature.loc[src_nature["status"] == "Supplier",  "id_src"])

    # ── Conqueror: attack enemy/neutral planets ──────────────────────────────
    attacks_conqueror = pd.DataFrame()
    conqueror_needs = None
    if conqueror_ids:

        _c_1_or_2 = (
            attacks_with_angle[attacks_with_angle["id_src"].isin(conqueror_ids)]
            .merge(top5_ids[["id_src", "id", "is_top5"]], on=["id_src", "id"], how="left")
            .assign(is_top5=lambda d: d["is_top5"].fillna(False))
            .query("is_top5")
            .loc[lambda d: d["owner"] != player_id]
            .assign(ships_needed=lambda d: np.where(
                d["owner"] == -1, d["ships"], d["ships"] + d["production"]
            ))
            # .loc[lambda d:
            #     (d["ships_needed"] + 1 <= d["ships_sent"]) &
            #     (d["ships_sent"] <= d["ships_needed"] + d["production_src"] + 1)
            # ]
            # .sort_values(["step", "ships_sent"])
            # .groupby(["id_src", "id"], sort=False).first().reset_index()
            # .assign(time_cost=lambda d: d["ships_needed"] / d["production_src"])
        )


        _c = (
            _c_1_or_2
            # attacks_with_angle[attacks_with_angle["id_src"].isin(conqueror_ids)]
            # .merge(top5_ids[["id_src", "id", "is_top5"]], on=["id_src", "id"], how="left")
            # .assign(is_top5=lambda d: d["is_top5"].fillna(False))
            # .query("is_top5")
            # .loc[lambda d: d["owner"] != player_id]
            # .assign(ships_needed=lambda d: np.where(
            #     d["owner"] == -1, d["ships"], d["ships"] + d["production"]
            # ))
            .loc[lambda d:
                (d["ships_needed"] + 1 <= d["ships_sent"]) &
                (d["ships_sent"] <= d["ships_needed"] + d["production_src"] + 1)
            ]
            .sort_values(["step", "ships_sent"])
            .groupby(["id_src", "id"], sort=False).first().reset_index()
            .assign(
                time_cost=lambda d: d["ships_needed"] / d["production_src"],
                score=lambda d: d["production"] / (d["time_cost"] + d["step_diff"]),
            )
        )
        if not _c.empty:
            conqueror_needs = (
                _c
                .groupby("id_src", sort=False)
                .agg(
                    ship_min=("ships_min", "min"),
                    all_need=("ships_sent", "sum"),
                    lowest_need=("ships_sent", "min"),
                    nb_need=("ships_sent", "count"),
                    max_score=("score", "max")
                )
                .reset_index()
            )
            attacks_conqueror = (
                _c
                # .assign(
                #     total_time_cost=_c.groupby("id_src")["time_cost"].transform("sum")
                # ).assign(
                #     score=lambda d: (
                #         (d["total_time_cost"] - d["time_cost"] - d["step_diff"]) * d["production"]
                #     )
                # )
                # .loc[lambda d: d["score"] > 0]
                .sort_values("score", ascending=False)
                .groupby("id_src", sort=False).first().reset_index()
                .loc[lambda d: d["ships_sent"] <= d["ships_min"]]
            )
            if not _c_1_or_2.empty:
                attacks_conqueror_2 = (
                    _c_1_or_2
                    .merge(
                        _c_1_or_2,
                        on="id",
                        how="inner",
                        suffixes=("", "_2"), # _2 is the ship to be sent later
                    )
                    .query("id_src != id_src_2")
                    .query("step < step_2")
                    .loc[lambda d:
                        (np.maximum(d["ships_needed"], d["ships_needed_2"]) + 1 <= d["ships_sent"] + d["ships_sent_2"]) &
                        (d["ships_sent"] + d["ships_sent_2"] <= np.maximum(d["ships_needed"], d["ships_needed_2"]) + d["production_src_2"] + 1)
                    ]
                    .loc[lambda d: d["ships_sent"] <= d["ships_min"]]
                    .loc[lambda d: d["ships_sent_2"] <= d["ships_min_2"] +  d["production_src_2"]]
                    .merge(
                        conqueror_needs[["id_src", "max_score"]],
                        on="id_src",
                        how="left"
                    )            
                    .merge(
                        conqueror_needs[["id_src", "max_score"]].rename(columns={"id_src": "id_src_2"}),
                        on="id_src_2",
                        how="left",
                        suffixes=("", "_2"),
                    )
                    .assign(
                        time_cost=lambda d: d["ships_sent"] / d["production_src"],
                        time_cost_2=lambda d: d["ships_sent_2"] / d["production_src_2"],
                        score=lambda d: d["production"] / (d["step_diff_2"] + (d["time_cost"]**2 + d["time_cost_2"]**2)**0.5),
                    )
                    .query("score > max_score and score > max_score_2")
                    .sort_values(["step_2", "ships_sent_2"])
                    .groupby(["id_src", "id"], sort=False).first().reset_index()
                    .sort_values(["step_2", "ships_sent_2"])
                    .pipe(lambda d: d.head(1) if d is not None and not d.empty else None)
                )


    # ── Supplier: reinforce own planets ─────────────────────────────────────
    attacks_supplier = pd.DataFrame()
    if supplier_ids and conqueror_needs is not None:
        _s = (
            attacks_with_angle[attacks_with_angle["id_src"].isin(supplier_ids)]
            .merge(top5_ids[["id_src", "id", "is_top5"]], on=["id_src", "id"], how="left")
            .assign(is_top5=lambda d: d["is_top5"].fillna(False))
            .loc[lambda d: d["is_top5"]]
            .assign(target_is_supplier=lambda d: d["id"].isin(supplier_ids))
            .query("not target_is_supplier")
            .merge(
                conqueror_needs,
                left_on="id",
                right_on="id_src",
                how="right"
            )
            .query("(lowest_need - ships_min) * 1.5 < ships_sent")
            .query("ships_min * 0.75 < ships_sent < ships_min")
            .sort_values(["all_need", "ships_sent"], ascending=[False, True])
            .groupby(["id_src"], sort=False).first().reset_index()
        )
        attacks_supplier = _s

    # ── Combine and emit ─────────────────────────────────────────────────────
    parts = [df for df in [attacks_conqueror, attacks_conqueror_2, attacks_supplier] if df is not None and not df.empty]
    if not parts:
        return moves

    attacks = pd.concat(parts, ignore_index=True)
    print("Currently using testing _04_score_and_decide")
    for _, row in attacks.iterrows():
        print(f"From {row['id_src']}, To {row['id']} at step {row['step']} "
            f"with {row['ships_sent']} ships (target has min {row['ships_min']})")

    moves += attacks[["id_src", "final_angle", "ships_sent"]].values.tolist()
    return moves

## Test 01 — We can attack together, but you wait to attack alone

- `id=0` (blue, 20 ships, +2): x=5, y=5
- `id=1` (blue, 20 ships, +2): x=10, y=5
- `id=2` (neutral, 35 ships, +2): x=7.5, y=10  ← joint target (neither alone: need 35, have 20)
- `id=3` (neutral, 21 ships, +2): x=15, y=5   ← solo target (need 21, have 20 → still short by 1)

Expected: 2-planet joint attack on id=2 (both planets contribute), no solo attack on id=3.

In [4]:
obs01 = Obs(
    planets=[
        [0,  0,  5.0,  5.0, 1 + math.log(2), 20, 2],  # Conqueror 1
        [1,  0, 10.0,  5.0, 1 + math.log(2), 20, 2],  # Conqueror 2
        [2, -1,  7.5, 10.0, 1 + math.log(2), 35, 2],  # Joint target  (ships_needed=35)
        [3, -1, 15.0,  5.0, 1 + math.log(2), 21, 2],  # Solo target   (ships_needed=21)
    ],
    angular_velocity=0.0,
)
df_s01, pd01 = StrategyPipeline._01_get_obs_dataframe(obs01, step=0, num_agents=2)
df_s01

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,5.0,5.0,1.693147,20,2,0,fix
1,0,1,10.0,5.0,1.693147,20,2,0,fix
2,0,2,7.5,10.0,1.693147,35,2,-1,fix
3,0,3,15.0,5.0,1.693147,21,2,-1,fix
4,1,0,5.0,5.0,1.693147,22,2,0,fix
5,1,1,10.0,5.0,1.693147,22,2,0,fix
6,1,2,7.5,10.0,1.693147,35,2,-1,fix
7,1,3,15.0,5.0,1.693147,21,2,-1,fix
8,2,0,5.0,5.0,1.693147,24,2,0,fix
9,2,1,10.0,5.0,1.693147,24,2,0,fix


In [5]:
pa01 = StrategyPipeline._02_get_all_opportunities(df_s01, pd01, player_id=0)
pa01

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,0.0,5.0,5.0,3.306853,3.355457,0.000000,9.837293e-02,6.184812,0.098373,0.0
1,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,0.0,5.0,5.0,3.306853,3.453664,0.000000,1.661470e-01,6.117038,0.166147,0.0
2,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,0.0,5.0,5.0,3.306853,3.540712,0.000000,2.044214e-01,6.078764,0.204421,0.0
3,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,0.0,5.0,5.0,3.306853,3.618966,0.000000,2.307850e-01,6.052400,0.230785,0.0
4,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,0.0,5.0,5.0,3.306853,4.383600,0.000000,3.384501e-01,5.944735,0.338450,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
623,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,0.0,10.0,10.0,11.013026,11.693147,0.129366,1.490116e-08,6.153819,0.129366,0.0
624,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,0.0,10.0,10.0,9.905649,11.064578,0.170060,1.252473e-01,6.113126,0.170060,0.0
625,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,0.0,10.0,10.0,11.064578,11.693147,0.125247,1.490116e-08,6.157938,0.125247,0.0
626,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,0.0,10.0,10.0,9.793147,10.793147,0.170017,1.441117e-01,6.113169,0.170017,0.0


In [6]:
safe01 = StrategyPipeline._03_filter_collision(pa01)
safe01

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,5.00000,5.00000,3.306853,3.355457,0.000000,0.098373,6.184812,0.098373,0.000000,0.000000
1,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,5.00000,5.00000,3.306853,3.453664,0.000000,0.166147,6.117038,0.166147,0.000000,0.000000
2,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,5.00000,5.00000,3.306853,3.540712,0.000000,0.204421,6.078764,0.204421,0.000000,0.000000
3,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,5.00000,5.00000,3.306853,3.618966,0.000000,0.230785,6.052400,0.230785,0.000000,0.000000
4,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,5.00000,5.00000,3.306853,4.383600,0.000000,0.338450,5.944735,0.338450,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,5.59017,5.59017,5.793147,6.793147,0.296465,0.193648,0.810684,1.403614,1.107149,1.107149
523,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,5.59017,5.59017,6.428863,7.283317,0.245968,0.000000,0.861181,1.353117,1.107149,1.107149
524,0,0,5.0,5.0,1.693147,20,2,fix,0,11,...,5.59017,5.59017,7.061649,7.283317,0.133404,0.000000,0.973745,1.240553,1.107149,1.107149
525,1,0,10.0,5.0,1.693147,20,2,fix,0,11,...,5.59017,5.59017,6.793147,7.283317,0.193648,0.000000,1.840796,2.228092,2.034444,2.034444


In [7]:
action01 = _04_score_and_decide(safe01, player_id=0)
print("Action:", action01)
snaps01 = simulate_with_action(copy.deepcopy(obs01), action01, 30)
make_animation(snaps01, title='Test 01 — Joint attack, wait alone', interval=200)

Action: []


## Test 02 — Am I attacking the next time step?

Same layout as Test 01 but each blue planet has **22 ships** (+2 each).

- `id=0` (blue, 22 ships, +2): x=5, y=5
- `id=1` (blue, 22 ships, +2): x=10, y=5
- `id=2` (neutral, 35 ships, +2): x=7.5, y=10  ← still needs joint (ships_needed=35, 22 < 35)
- `id=3` (neutral, 21 ships, +2): x=15, y=5   ← now solo-capturable (ships_needed=21, 22 > 21)

Expected: id=0 or id=1 attacks id=3 solo this step (ROI is immediate); joint combo on id=2 may also fire.

In [8]:
obs02 = Obs(
    planets=[
        [0,  0,  5.0,  5.0, 1 + math.log(2), 22, 2],  # Conqueror 1
        [1,  0, 10.0,  5.0, 1 + math.log(2), 22, 2],  # Conqueror 2
        [2, -1,  7.5, 10.0, 1 + math.log(2), 35, 2],  # Joint target  (ships_needed=35)
        [3, -1, 15.0,  5.0, 1 + math.log(2), 21, 2],  # Solo target   (ships_needed=21)
    ],
    angular_velocity=0.0,
)
df_s02, pd02 = StrategyPipeline._01_get_obs_dataframe(obs02, step=0, num_agents=2)
df_s02

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,5.0,5.0,1.693147,22,2,0,fix
1,0,1,10.0,5.0,1.693147,22,2,0,fix
2,0,2,7.5,10.0,1.693147,35,2,-1,fix
3,0,3,15.0,5.0,1.693147,21,2,-1,fix
4,1,0,5.0,5.0,1.693147,24,2,0,fix
5,1,1,10.0,5.0,1.693147,24,2,0,fix
6,1,2,7.5,10.0,1.693147,35,2,-1,fix
7,1,3,15.0,5.0,1.693147,21,2,-1,fix
8,2,0,5.0,5.0,1.693147,26,2,0,fix
9,2,1,10.0,5.0,1.693147,26,2,0,fix


In [9]:
pa02 = StrategyPipeline._02_get_all_opportunities(df_s02, pd02, player_id=0)
pa02

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,0.0,5.0,5.0,3.306853,3.355457,0.000000,9.837293e-02,6.184812,0.098373,0.0
1,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,0.0,5.0,5.0,3.306853,3.453664,0.000000,1.661470e-01,6.117038,0.166147,0.0
2,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,0.0,5.0,5.0,3.306853,3.540712,0.000000,2.044214e-01,6.078764,0.204421,0.0
3,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,0.0,5.0,5.0,3.306853,3.618966,0.000000,2.307850e-01,6.052400,0.230785,0.0
4,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,0.0,5.0,5.0,3.306853,3.690114,0.000000,2.504135e-01,6.032772,0.250414,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,0.0,10.0,10.0,11.013026,11.693147,0.129366,1.490116e-08,6.153819,0.129366,0.0
648,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,0.0,10.0,10.0,9.905649,11.064578,0.170060,1.252473e-01,6.113126,0.170060,0.0
649,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,0.0,10.0,10.0,11.064578,11.693147,0.125247,1.490116e-08,6.157938,0.125247,0.0
650,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,0.0,10.0,10.0,9.793147,10.793147,0.170017,1.441117e-01,6.113169,0.170017,0.0


In [10]:
safe02 = StrategyPipeline._03_filter_collision(pa02)
safe02

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,5.00000,5.00000,3.306853,3.355457,0.000000,0.098373,6.184812,0.098373,0.000000,0.000000
1,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,5.00000,5.00000,3.306853,3.453664,0.000000,0.166147,6.117038,0.166147,0.000000,0.000000
2,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,5.00000,5.00000,3.306853,3.540712,0.000000,0.204421,6.078764,0.204421,0.000000,0.000000
3,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,5.00000,5.00000,3.306853,3.618966,0.000000,0.230785,6.052400,0.230785,0.000000,0.000000
4,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,5.00000,5.00000,3.306853,3.690114,0.000000,0.250414,6.032772,0.250414,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
542,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,5.59017,5.59017,5.793147,6.793147,0.296465,0.193648,0.810684,1.403614,1.107149,1.107149
543,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,5.59017,5.59017,6.428863,7.283317,0.245968,0.000000,0.861181,1.353117,1.107149,1.107149
544,0,0,5.0,5.0,1.693147,22,2,fix,0,11,...,5.59017,5.59017,7.061649,7.283317,0.133404,0.000000,0.973745,1.240553,1.107149,1.107149
545,1,0,10.0,5.0,1.693147,22,2,fix,0,11,...,5.59017,5.59017,6.793147,7.283317,0.193648,0.000000,1.840796,2.228092,2.034444,2.034444


In [11]:
action02 = _04_score_and_decide(safe02, player_id=0)
print("Action:", action02)
snaps02 = simulate_with_action(copy.deepcopy(obs02), action02, 30)
make_animation(snaps02, title='Test 02 — Solo attack available', interval=200)

Currently using testing _04_score_and_decide
From 1, To 3 at step 1 with 22 ships (target has min 22)
Action: [[1.0, 0.0, 22.0]]
